In [1]:
# 1) Imports + Configs

from pathlib import Path
import numpy as np
import pandas as pd
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression

from sentence_transformers import SentenceTransformer

from xgboost import XGBClassifier
from scipy.sparse import hstack, csr_matrix

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [2]:
# 2) Load processed dataset v2

PROJECT_ROOT = Path("..").resolve()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

processed_path = DATA_PROCESSED / "jailbreak_benchmarks_processed_v2.csv"
df = pd.read_csv(processed_path)

print("Rows:", len(df))
print("\nSplit counts:")
print(df["split"].value_counts())
print("\nLabel counts:")
print(df["label"].value_counts())

Rows: 1647

Split counts:
split
ood_test    768
train       615
val         132
test        132
Name: count, dtype: int64

Label counts:
label
1    993
0    654
Name: count, dtype: int64


In [3]:
# Split views

df_train = df[df["split"] == "train"].copy()
df_val   = df[df["split"] == "val"].copy()
df_test  = df[df["split"] == "test"].copy()
df_ood   = df[df["split"] == "ood_test"].copy()

for name, d in [("train", df_train), ("val", df_val), ("test", df_test), ("ood_test", df_ood)]:
    print(f"{name:8s}", d.shape, d["label"].value_counts().to_dict())


train    (615, 8) {1: 426, 0: 189}
val      (132, 8) {1: 92, 0: 40}
test     (132, 8) {1: 91, 0: 41}
ood_test (768, 8) {1: 384, 0: 384}


In [4]:
# 3) Build M2 Features: TF-IDF + FLAGS (NO IBVS)

# TF–IDF
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=20000,
)
tfidf.fit(df_train["prompt_text"].astype(str))

X_lex_train = tfidf.transform(df_train["prompt_text"].astype(str))
X_lex_val   = tfidf.transform(df_val["prompt_text"].astype(str))
X_lex_test  = tfidf.transform(df_test["prompt_text"].astype(str))
X_lex_ood   = tfidf.transform(df_ood["prompt_text"].astype(str))

y_train = df_train["label"].values
y_val   = df_val["label"].values
y_test  = df_test["label"].values
y_ood   = df_ood["label"].values

In [5]:
# FLAGS
OVERRIDE_PATTERNS = [
    r"ignore (all )?(previous|prior) instructions",
    r"you are now",
    r"disregard (the )?(previous|above) rules",
    r"as an unfiltered model",
    r"system prompt",
    r"from now on, you must",
]

def lexical_flags(text: str) -> dict:
    if not isinstance(text, str):
        text = ""
    t = text.lower()
    return {
        "has_ignore_prev": bool(re.search(OVERRIDE_PATTERNS[0], t)),
        "has_you_are_now": "you are now" in t,
        "has_disregard": ("disregard" in t and "instructions" in t),
        "has_system_prompt": ("system prompt" in t),
        "len_chars": len(text),
        "len_tokens_approx": len(text.split()),
    }

def make_flag_matrix(df_split: pd.DataFrame):
    flags_df = pd.DataFrame([lexical_flags(t) for t in df_split["prompt_text"].astype(str)])
    return csr_matrix(flags_df.values.astype(float))

In [6]:
X_flag_train = make_flag_matrix(df_train)
X_flag_val   = make_flag_matrix(df_val)
X_flag_test  = make_flag_matrix(df_test)
X_flag_ood   = make_flag_matrix(df_ood)

# FUSE
X_m2_train = hstack([X_lex_train, X_flag_train]).tocsr()
X_m2_val   = hstack([X_lex_val,   X_flag_val]).tocsr()
X_m2_test  = hstack([X_lex_test,  X_flag_test]).tocsr()
X_m2_ood   = hstack([X_lex_ood,   X_flag_ood]).tocsr()

print("\nM2 feature shapes:", X_m2_train.shape, X_m2_val.shape, X_m2_test.shape, X_m2_ood.shape)



M2 feature shapes: (615, 1511) (132, 1511) (132, 1511) (768, 1511)


In [7]:
# 4) Train M2 Model (XGBoost)

m2 = XGBClassifier(
    objective="binary:logistic",
    n_estimators=400,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=RANDOM_SEED,
)

m2.fit(X_m2_train, y_train, eval_set=[(X_m2_val, y_val)], verbose=False)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=-1,
              num_parallel_tree=None, ...)

In [8]:
# 5) Train Semantic Model (BGE + Logistic Regression)

sem_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

def encode_texts(texts, batch_size=32):
    return sem_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

X_sem_train = encode_texts(df_train["prompt_text"].astype(str).tolist())
X_sem_val   = encode_texts(df_val["prompt_text"].astype(str).tolist())
X_sem_test  = encode_texts(df_test["prompt_text"].astype(str).tolist())
X_sem_ood   = encode_texts(df_ood["prompt_text"].astype(str).tolist())

sem_clf = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
sem_clf.fit(X_sem_train, y_train)

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

LogisticRegression(class_weight='balanced', max_iter=5000, n_jobs=-1,
                   random_state=42)

In [9]:
# 6) Hybrid Routing

# Rule:
#   if p_sem <= tau_low  -> predict benign (0)
#   elif p_sem >= tau_high -> predict harmful (1)
#   else -> fallback to M2 prediction


def hybrid_predict(X_sem, X_m2, tau_low=0.30, tau_high=0.70):
    p_sem = sem_clf.predict_proba(X_sem)[:, 1]
    y_fallback = m2.predict(X_m2)

    y_hat = np.empty_like(y_fallback)
    route = np.empty_like(y_fallback, dtype=object)

    # semantic confident benign
    mask_benign = p_sem <= tau_low
    y_hat[mask_benign] = 0
    route[mask_benign] = "semantic_low"

    # semantic confident harmful
    mask_harm = p_sem >= tau_high
    y_hat[mask_harm] = 1
    route[mask_harm] = "semantic_high"

    # uncertain -> fallback
    mask_uncertain = ~(mask_benign | mask_harm)
    y_hat[mask_uncertain] = y_fallback[mask_uncertain]
    route[mask_uncertain] = "fallback_m2"

    coverage_semantic = float((mask_benign | mask_harm).mean())
    defer_rate = float(mask_uncertain.mean())

    return y_hat, route, coverage_semantic, defer_rate

def eval_system(name, y_true, y_pred, route):
    acc = accuracy_score(y_true, y_pred)
    macro = f1_score(y_true, y_pred, average="macro")
    print(f"\n=== {name} (HYBRID) ===")
    print(classification_report(y_true, y_pred, digits=3))
    route_counts = pd.Series(route).value_counts(normalize=True).to_dict()
    print("Routing proportions:", {k: round(v, 4) for k, v in route_counts.items()})
    return {
        "split": name,
        "acc": float(acc),
        "macro_f1": float(macro),
        "semantic_coverage": float((pd.Series(route) != "fallback_m2").mean()),
        "defer_rate": float((pd.Series(route) == "fallback_m2").mean()),
    }

In [10]:
# Choose initial thresholds (we will sweep later)
TAU_LOW = 0.30
TAU_HIGH = 0.70

val_pred, val_route, val_cov, val_def = hybrid_predict(X_sem_val, X_m2_val, TAU_LOW, TAU_HIGH)
test_pred, test_route, _, _           = hybrid_predict(X_sem_test, X_m2_test, TAU_LOW, TAU_HIGH)
ood_pred, ood_route, _, _             = hybrid_predict(X_sem_ood, X_m2_ood, TAU_LOW, TAU_HIGH)

val_metrics  = eval_system("VAL",  y_val,  val_pred,  val_route)
test_metrics = eval_system("TEST", y_test, test_pred, test_route)
ood_metrics  = eval_system("OOD",  y_ood,  ood_pred,  ood_route)

summary = pd.DataFrame([val_metrics, test_metrics, ood_metrics])
summary.insert(0, "model", f"HYBRID_sem({TAU_LOW},{TAU_HIGH})_else_M2")
print("\n=== Hybrid Summary ===")
display(summary)


=== VAL (HYBRID) ===
              precision    recall  f1-score   support

           0      0.944     0.850     0.895        40
           1      0.938     0.978     0.957        92

    accuracy                          0.939       132
   macro avg      0.941     0.914     0.926       132
weighted avg      0.940     0.939     0.938       132

Routing proportions: {'semantic_high': 0.6061, 'semantic_low': 0.2197, 'fallback_m2': 0.1742}

=== TEST (HYBRID) ===
              precision    recall  f1-score   support

           0      0.919     0.829     0.872        41
           1      0.926     0.967     0.946        91

    accuracy                          0.924       132
   macro avg      0.923     0.898     0.909       132
weighted avg      0.924     0.924     0.923       132

Routing proportions: {'semantic_high': 0.6061, 'semantic_low': 0.2197, 'fallback_m2': 0.1742}

=== OOD (HYBRID) ===
              precision    recall  f1-score   support

           0      0.700     0.870   

,model,split,acc,macro_f1,semantic_coverage,defer_rate
0,"HYBRID_sem(0.3,0.7)_else_M2",VAL,0.939394,0.926092,0.825758,0.174242
1,"HYBRID_sem(0.3,0.7)_else_M2",TEST,0.924242,0.909016,0.825758,0.174242
2,"HYBRID_sem(0.3,0.7)_else_M2",OOD,0.748698,0.744958,0.523438,0.476562


In [11]:
# 7) Threshold Sweep (Small Grid)

# We sweep tau_low and tau_high and rank by OOD macro-F1 primarily,
# while also reporting how much we defer (coverage vs escalation trade-off).

grid_tau_low = [0.20, 0.25, 0.30, 0.35, 0.40]
grid_tau_high = [0.60, 0.65, 0.70, 0.75, 0.80]

rows = []
for tl in grid_tau_low:
    for th in grid_tau_high:
        if tl >= th:
            continue

        # evaluate on OOD only (primary), and keep defer rate
        ood_pred_g, ood_route_g, cov_g, def_g = hybrid_predict(X_sem_ood, X_m2_ood, tl, th)
        ood_acc = accuracy_score(y_ood, ood_pred_g)
        ood_macro = f1_score(y_ood, ood_pred_g, average="macro")

        rows.append({
            "tau_low": tl,
            "tau_high": th,
            "ood_acc": float(ood_acc),
            "ood_macro_f1": float(ood_macro),
            "semantic_coverage": float(cov_g),
            "defer_rate": float(def_g),
        })

sweep = pd.DataFrame(rows).sort_values(by="ood_macro_f1", ascending=False).reset_index(drop=True)
print("\n=== Threshold sweep (ranked by OOD macro-F1) ===")
display(sweep.head(15))


=== Threshold sweep (ranked by OOD macro-F1) ===


,tau_low,tau_high,ood_acc,ood_macro_f1,semantic_coverage,defer_rate
0,0.30,0.65,0.748698,0.745277,0.559896,0.440104
1,0.30,0.70,0.748698,0.744958,0.523438,0.476562
2,0.30,0.60,0.747396,0.744483,0.595052,0.404948
3,0.20,0.65,0.743490,0.741861,0.375000,0.625000
4,0.20,0.70,0.743490,0.741639,0.338542,0.661458
5,0.25,0.65,0.743490,0.741402,0.458333,0.541667
6,0.25,0.70,0.743490,0.741151,0.421875,0.578125
7,0.20,0.60,0.742188,0.740907,0.410156,0.589844
8,0.25,0.60,0.742188,0.740496,0.493490,0.506510
9,0.40,0.65,0.747396,0.739706,0.734375,0.265625
